# Survival LLM: Survival Q&A Fine-Tuning

Generate a survival Q&A dataset with the LightningRod SDK and fine-tune using the LightningRod SFT API.

**Pipeline:** LightningRod topic tree → Q&A generation → SFT fine-tuning

In [13]:
%pip install lightningrod-ai python-dotenv pandas -q

from IPython.display import clear_output
clear_output()

## Setup

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) for your API key.

In [14]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
lr = LightningRod(api_key=config.get_config_value("LIGHTNINGROD_API_KEY"))

## 1. Define survival domains

Define root topics for the `TopicTreeSeedGenerator`, which recursively decomposes each domain into specific subtopics. Each leaf path becomes a seed for Q&A generation.

In [15]:
DOMAINS = [
    "Field medicine and trauma care in austere environments",
    "Water purification and safe water sourcing without electricity",
    "Food preservation, canning, and long-term storage without refrigeration",
    "Ham radio and emergency communications setup and operation",
    "Land navigation using map, compass, and natural indicators",
    "Growing food: gardening, permaculture, and seed saving",
    "Herbal medicine and natural remedies from wild plants",
    "Construction, structural repair, and improvised building",
    "Welding, metalworking, and tool fabrication",
    "Vehicle repair and mechanical troubleshooting without a shop",
    "Fire starting, fire management, and fuel sourcing",
    "Emergency shelter building from natural and salvaged materials",
    "Hunting, trapping, fishing, and wild game processing",
    "Knot tying, rope work, and cordage making",
    "Weather reading and natural forecasting without instruments",
    "Perimeter security, self-defense, and community safety planning",
]

In [16]:
from lightningrod import TopicTreeSeedGenerator

# degree=5, depth=3 → 125 paths/domain → 2000 total topics → 20k questions at 10/seed
# Start small for testing; scale up DOMAINS[:] and tree params for production runs.
TREE_DEGREE = 3
TREE_DEPTH = 2

seed_generator = TopicTreeSeedGenerator(
    topic=DOMAINS[:2],
    tree_depth=TREE_DEPTH,
    tree_degree=TREE_DEGREE,
    model_system_prompt=(
        "You are an expert in survival and self-reliance. "
        "Generate specific, practical subtopics useful in a grid-down emergency."
    ),
)

## 2. Generate Q&A dataset

Generate questions with `QuestionGenerator` and label answers with `WebSearchLabeler` for web-grounded accuracy. The `TopicTreeSeedGenerator` feeds topics directly into the pipeline.

In [ ]:
from lightningrod import (
    FreeResponseAnswerType, QuestionGenerator,
    QuestionPipeline, WebSearchLabeler,
)

answer_type = FreeResponseAnswerType(
    labeler_instruction=(
        "You are a survival expert giving emergency field instructions. "
        "Give direct, numbered step-by-step instructions. No introductions, disclaimers, "
        "or filler. Start with the first action. Use specific measurements and techniques. "
        "Assume no professional help, stores, or infrastructure available."
    ),
    answer_format_instruction=(
        "Provide a direct, step-by-step survival answer. No introduction — start with step 1. "
        "Use specific measurements and techniques. Provide your answer between <answer></answer> tags."
    ),
    question_generation_instruction=(
        "Generate specific, practical how-to questions about survival techniques for "
        "grid-down emergencies. Each question must ask HOW to perform a specific procedure "
        "with limited or no modern tools. Each must cover a UNIQUE technique."
    ),
)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        questions_per_seed=10,
        instructions=(
            "Generate practical, actionable survival questions for grid-down emergencies. "
            "Questions must be specific, scenario-based, and ask HOW to do something with "
            "limited or no modern tools. Each must cover a DISTINCT technique."
        ),
        examples=[
            "How do I purify water using only sand, gravel, and charcoal when I have no commercial filter?",
            "What are the signs of a tension pneumothorax and how do I perform a needle decompression in the field?",
            "How do I build a Dakota fire hole to minimize visible smoke and maximize heat efficiency?",
            "How do I make a basic antenna for a Baofeng UV-5R to extend its range in mountainous terrain?",
        ],
        bad_examples=[
            "What is survival? (too vague)",
            "Tell me about water purification. (not actionable)",
            "How does a ham radio work? (theoretical, not a how-to)",
        ],
    ),
    labeler=WebSearchLabeler(answer_type=answer_type, confidence_threshold=0.8),
)

dataset = lr.transforms.run(pipeline, name="SurvivalLLM", max_seeds=20)  # Increase to ~1000 for a real run

result_samples = dataset.download()
valid = sum(1 for s in result_samples if s.is_valid)
print(f"{len(result_samples)} samples, {valid} valid ({valid/len(result_samples)*100:.0f}%)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           378c0c0b-aa60-4b73-ade0-87d4f4141e5e                                                       │
│                                                                                                                 │
│    Total cost: $0.00                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃ In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons   ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ TopicTreeSeedGene… │ Complete             │  1 │   9 │        0 │      0 │ -                   │       1s │  │
│  │ QuestionGenerator… │ Complete             │  9 │  90 │        0 │      0 │ -                   │       0s │  │
│  │ WebSearchLabelerT… │ Complete             │ 90 │  88 │        2 │      0 │ Undetermined label  │       1s │  │
│  │                    │                      │    │     │          │        │ (2)                 │          │  │
│  └────────────────────┴──────────────────────┴────┴─────┴──────────┴────────┴─────────────────────┴──────────┘  │
│                                                                                                                 │
│    View full details:                                                                                           │
│  ]8;id=918748;https://dashboard.lightningrod.ai/?redirect=/datasets/801ef413-6262-4d01-b896-5875e75f06a8\https://dashboard.lightningrod.ai/?redirect=/datasets/801ef413-6262-4d01-b896-5875e75f06a8]8;;\                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

90 samples, 88 valid (98%)


In [18]:
import pandas as pd

rows = [
    {
        "question": s.question.question_text if s.question else None,
        "answer": s.label.label if s.label else None,
        "confidence": s.label.label_confidence if s.label else None,
    }
    for s in result_samples if s.is_valid
]
pd.DataFrame(rows).head()

,question,answer,confidence
0,How do I use a sewing needle and a magnet to c...,1. Magnetize the needle: Stroke the sewing nee...,1.00
1,How do I determine the safe flow rate of a gra...,1. Perform a Visual Integrity Check: Examine e...,0.95
2,How do I create a makeshift Faraday cage using...,1. Wrap the electronic device in a non-conduct...,1.00
3,How do I convert a rotary hand-well pump into ...,1. Source a 12V DC windshield wiper motor for ...,0.95
4,How do I create a bow drill set from found har...,"1. Select Materials: Locate dead, dry softwood...",1.00


## 3. Prepare training data

Split the dataset into train and test sets for fine-tuning.

In [19]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(),
    split=SplitParams(test_size=0.2, strategy="random"),
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 90 samples                                                                                     │
│                                                                                                                 │
│    Filter:  Dropped 2 invalid → 88 remain                                                                       │
│    Dedup:   88 remain (0 duplicates)                                                                            │
│    Split:   Splits: 70 train | 18 test (0 dropped, no prediction_date)                                          │
│                                                                                                                 │
│  ⚠ Unhealthy dataset                                                                                            │
│                                                                                                                 │
│  Only 70 train samples remain after preparation. This is below the recommended minimum of +1000 for effective   │
│  training.                                                                                                      │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Increase max_questions in lr.transforms.run() to generate more samples.                                  │
│      • Increase questions_per_seed in your question generator (ForwardLookingQuestionGenerator or               │
│  QuestionGenerator) to produce more questions from each seed article.Add more search queries to your seed       │
│  generator to diversify seed sources.                                                                           │
│      • Widen the seed generator date range (start_date to end_date) to capture more events.                     │
│                                                                                                                 │
│  Only 18 test samples remain after preparation. This is below the recommended minimum of +200 for reliable      │
│  evaluation.                                                                                                    │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Generate more samples overall — test samples come from the most recent portion of your date range.       │
│      • Ensure your seed generator date range extends close to the present so recent events appear in the test   │
│  set.                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 4. Fine-tune with LightningRod SFT

LoRA SFT on the generated dataset using the LightningRod hosted training API.

In [20]:
from lightningrod import SFTTrainingConfig
from lightningrod import training

sft_config = SFTTrainingConfig(
    base_model_id="openai/gpt-oss-120b",
    training_steps=50,
    epochs=3,
    learning_rate=2e-4,
    lora_rank=16,
)

job = lr.training.run(sft_config, dataset=train_dataset, name="SurvivalLLM")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job ID: 1aa11dce-eed0-44aa-9180-ba289a2e794f                                                                 │
│                                                                                                                 │
│    Model: checkpoint:1aa11dce-eed0-44aa-9180-ba289a2e794f                                                       │
│                                                                                                                 │
│    loss: latest 1.7754  avg 1.8139  (9 steps)  (lower is better)                                                │
│        █▆▅▂▂▃▁▁▃                                                                                                │
│    learning_rate: latest 0.0000  avg 0.0001  (9 steps)                                                          │
│        ██▇▆▅▄▃▂▁                                                                                                │
│    mean_train_tokens: latest 517.6667  avg 523.7118  (9 steps)                                                  │
│        ▃█▁▃█▁▃█▁                                                                                                │
│                                                                                                                 │
│    Cost:  $0.06                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 5. Evaluate: base vs fine-tuned

In [ ]:
SYSTEM_PROMPT = (
    "You are SurvivalLLM. Give direct, step-by-step survival instructions. "
    "No introductions or disclaimers. Start with the first action. "
    "Be specific with measurements and techniques."
)

test_questions = [
    "How do I purify water using only materials I can find in a forest?",
    "How do I stop severe arterial bleeding with no medical supplies?",
    "How do I start a fire in wet conditions with no matches?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    ft = lr.predict(q, model=job.model_id, system_prompt=SYSTEM_PROMPT)
    print(f"FINE-TUNED:  {ft.content[:300]}")